# LeetCode 1664 — Ways to Make a Fair Array bằng Python

Link đề: https://leetcode.com/problems/ways-to-make-a-fair-array/ · Độ khó: **Medium** · Chủ đề: Array, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def waysToMakeFair(self, nums: List[int]) -> int:
        ...
```

Notebook này đi từ cách **vét cạn** (xóa thử từng chỉ số rồi tính lại tổng) tới các cách **tổng tiền tố** chạy `O(n)` và cuối cùng là cách **một biến tổng đan dấu** gọn nhất.

## 1. Mục tiêu học

1. Hiểu chính xác điều gì xảy ra với **chỉ số** của các phần tử khi xóa một phần tử khỏi mảng: bên trái giữ nguyên, bên phải **lùi 1** nên **đảo chẵn/lẻ**.
2. Thành thạo **tổng tiền tố theo chẵn/lẻ** (`prefix_even`, `prefix_odd`) và cách lấy tổng một đoạn trong `O(1)`.
3. Đi từ vét cạn `O(n²)` → mảng tiền tố `O(n)`/`O(n)` → hai cặp tổng chạy `O(n)`/`O(1)` → **tổng đan dấu** một biến.
4. Biết cách viết hàm `trace_*` in trạng thái từng bước để tự kiểm tra lập luận bằng tay.
5. Viết bộ test ngẫu nhiên có seed cố định, đối chiếu mọi cách với brute force, đúng ràng buộc đề.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`. Bạn chọn **đúng một** chỉ số (đánh số từ 0) và **xóa** phần tử tại đó. Chú ý: sau khi xóa, chỉ số của các phần tử phía sau sẽ **thay đổi**.

Ví dụ với `nums = [6,1,7,4,1]`:
- Xóa chỉ số `1` → `[6,7,4,1]`
- Xóa chỉ số `2` → `[6,1,4,1]`
- Xóa chỉ số `4` → `[6,1,7,4]`

Một mảng gọi là **fair** (cân bằng) nếu **tổng các phần tử ở chỉ số lẻ** bằng **tổng các phần tử ở chỉ số chẵn**.

Hãy trả về **số lượng** chỉ số mà nếu xóa nó thì mảng còn lại là fair.

**Ví dụ 1:** `nums = [2,1,6,4]` → `1`
- Xóa 0: `[1,6,4]` → chẵn `1+4=5`, lẻ `6` → không fair.
- Xóa 1: `[2,6,4]` → chẵn `2+4=6`, lẻ `6` → **fair**.
- Xóa 2: `[2,1,4]` → chẵn `6`, lẻ `1` → không.
- Xóa 3: `[2,1,6]` → chẵn `8`, lẻ `1` → không.

**Ví dụ 2:** `nums = [1,1,1]` → `3` (xóa chỉ số nào cũng còn `[1,1]`, fair).

**Ví dụ 3:** `nums = [1,2,3]` → `0`.

**Ràng buộc:**
- `1 ≤ nums.length ≤ 10⁵`
- `1 ≤ nums[i] ≤ 10⁴`

In [1]:
import random
import time
from typing import List
from itertools import accumulate

examples = [
    ([2, 1, 6, 4], 1),
    ([1, 1, 1], 3),
    ([1, 2, 3], 0),
]

def is_fair(arr):
    """Mảng fair nếu tổng chỉ số chẵn == tổng chỉ số lẻ."""
    return sum(arr[0::2]) == sum(arr[1::2])

for nums, expected in examples:
    print(f"nums = {nums}  -> kỳ vọng {expected}")
    for i in range(len(nums)):
        rest = nums[:i] + nums[i + 1:]
        print(f"   xóa {i}: {rest}  chẵn={sum(rest[0::2])}, lẻ={sum(rest[1::2])}, fair={is_fair(rest)}")

nums = [2, 1, 6, 4]  -> kỳ vọng 1
   xóa 0: [1, 6, 4]  chẵn=5, lẻ=6, fair=False
   xóa 1: [2, 6, 4]  chẵn=6, lẻ=6, fair=True
   xóa 2: [2, 1, 4]  chẵn=6, lẻ=1, fair=False
   xóa 3: [2, 1, 6]  chẵn=8, lẻ=1, fair=False
nums = [1, 1, 1]  -> kỳ vọng 3
   xóa 0: [1, 1]  chẵn=1, lẻ=1, fair=True
   xóa 1: [1, 1]  chẵn=1, lẻ=1, fair=True
   xóa 2: [1, 1]  chẵn=1, lẻ=1, fair=True
nums = [1, 2, 3]  -> kỳ vọng 0
   xóa 0: [2, 3]  chẵn=2, lẻ=3, fair=False
   xóa 1: [1, 3]  chẵn=1, lẻ=3, fair=False
   xóa 2: [1, 2]  chẵn=1, lẻ=2, fair=False


## 3. Kiến thức nền

### 3.1 Cắt lát (slicing) với bước nhảy

`arr[0::2]` lấy các phần tử ở chỉ số `0, 2, 4, …` (chỉ số chẵn); `arr[1::2]` lấy chỉ số `1, 3, 5, …` (chỉ số lẻ). Mỗi lát cắt tạo **list mới**, tốn `O(n)` thời gian và bộ nhớ.

In [2]:
arr = [6, 1, 7, 4, 1]
print("chỉ số chẵn:", arr[0::2], "tổng =", sum(arr[0::2]))
print("chỉ số lẻ  :", arr[1::2], "tổng =", sum(arr[1::2]))
print("đảo ngược  :", arr[::-1])

chỉ số chẵn: [6, 7, 1] tổng = 14
chỉ số lẻ  : [1, 4] tổng = 5
đảo ngược  : [1, 4, 7, 1, 6]


### 3.2 `enumerate` và phép `% 2`

`enumerate(arr)` cho từng cặp `(chỉ số, giá trị)`. `i % 2` bằng `0` nếu `i` chẵn, `1` nếu `i` lẻ — rất tiện để chọn "ô" trong một list 2 phần tử: `s[i % 2] += x`.

In [3]:
s = [0, 0]          # s[0] = tổng chỉ số chẵn, s[1] = tổng chỉ số lẻ
for i, x in enumerate(arr):
    s[i % 2] += x
    print(f"i={i}, x={x}, i%2={i % 2}, s={s}")

i=0, x=6, i%2=0, s=[6, 0]
i=1, x=1, i%2=1, s=[6, 1]
i=2, x=7, i%2=0, s=[13, 1]
i=3, x=4, i%2=1, s=[13, 5]
i=4, x=1, i%2=0, s=[14, 5]


### 3.3 Tổng tiền tố (prefix sum)

`pre[k] = arr[0] + … + arr[k-1]` (với `pre[0] = 0`). Khi đó tổng đoạn `arr[l..r]` (bao gồm hai đầu) là `pre[r+1] - pre[l]` — tính trong `O(1)` sau khi chuẩn bị `O(n)`.

`itertools.accumulate(arr, initial=0)` dựng sẵn mảng tiền tố.

In [4]:
pre = list(accumulate(arr, initial=0))
print("pre =", pre)
l, r = 1, 3
print(f"tổng arr[{l}..{r}] = pre[{r + 1}] - pre[{l}] =", pre[r + 1] - pre[l], "| kiểm tra:", sum(arr[l:r + 1]))

pre = [0, 6, 7, 14, 18, 19]
tổng arr[1..3] = pre[4] - pre[1] = 12 | kiểm tra: 12


### 3.4 Tổng tiền tố **theo chẵn/lẻ**

Ta cần tổng các phần tử chỉ số chẵn **trong một đoạn**. Dựng hai mảng:
- `pe[k]` = tổng `arr[j]` với `j < k` và `j` chẵn,
- `po[k]` = tổng `arr[j]` với `j < k` và `j` lẻ.

Tổng phần tử chỉ số chẵn trong đoạn `[l, r]` là `pe[r+1] - pe[l]`.

In [5]:
def parity_prefix(a):
    pe, po = [0], [0]
    for j, x in enumerate(a):
        pe.append(pe[-1] + (x if j % 2 == 0 else 0))
        po.append(po[-1] + (x if j % 2 == 1 else 0))
    return pe, po

pe, po = parity_prefix(arr)
print("pe =", pe)
print("po =", po)
print("tổng chỉ số chẵn trong [1..4]:", pe[5] - pe[1], "| kiểm tra:", sum(arr[j] for j in range(1, 5) if j % 2 == 0))

pe = [0, 6, 6, 13, 13, 14]
po = [0, 0, 1, 1, 5, 5]
tổng chỉ số chẵn trong [1..4]: 8 | kiểm tra: 8


### 3.5 Tổng **đan dấu** (alternating sum)

`alt = arr[0] - arr[1] + arr[2] - arr[3] + …` = (tổng chỉ số chẵn) − (tổng chỉ số lẻ). Mảng fair ⇔ `alt == 0`. Đây là "một con số" gói gọn điều kiện fair, sẽ dùng ở Cách 4.

In [6]:
def alt_sum(a):
    return sum(x if j % 2 == 0 else -x for j, x in enumerate(a))

for a in ([2, 6, 4], [1, 6, 4], [1, 1], [6, 1, 7, 4, 1]):
    print(a, "alt =", alt_sum(a), "fair =", alt_sum(a) == 0)

[2, 6, 4] alt = 0 fair = True
[1, 6, 4] alt = -1 fair = False
[1, 1] alt = 0 fair = True
[6, 1, 7, 4, 1] alt = 9 fair = False


## 4. Ý tưởng nền tảng / trực giác — tính bằng tay

Lấy `nums = [2,1,6,4]`, xóa chỉ số `i = 1` (giá trị `1`):

| vị trí cũ | 0 | 1 | 2 | 3 |
|---|---|---|---|---|
| giá trị | 2 | **1 (xóa)** | 6 | 4 |
| vị trí mới | 0 | — | 1 | 2 |
| chẵn/lẻ cũ → mới | chẵn → chẵn | — | chẵn → **lẻ** | lẻ → **chẵn** |

Quan sát:
- Phần tử **bên trái** `i` giữ nguyên chỉ số ⇒ giữ nguyên chẵn/lẻ.
- Phần tử **bên phải** `i` lùi 1 ô ⇒ **đổi** chẵn ↔ lẻ.

Vậy sau khi xóa `i`:
- **Tổng chẵn mới** = (tổng chẵn bên trái) + (tổng **lẻ** bên phải)
- **Tổng lẻ mới** = (tổng lẻ bên trái) + (tổng **chẵn** bên phải)

Với `i = 1`: trái = `[2]` (chẵn 2, lẻ 0); phải = `[6 (chẵn cũ), 4 (lẻ cũ)]`. Tổng chẵn mới = `2 + 4 = 6`, tổng lẻ mới = `0 + 6 = 6` ⇒ fair. Khớp với đề.

In [7]:
def split_view(nums, i):
    left, right = nums[:i], nums[i + 1:]
    left_even = sum(left[0::2]); left_odd = sum(left[1::2])
    # các phần tử bên phải: chỉ số cũ là i+1, i+2, ...
    right_even_old = sum(x for j, x in enumerate(nums) if j > i and j % 2 == 0)
    right_odd_old = sum(x for j, x in enumerate(nums) if j > i and j % 2 == 1)
    new_even = left_even + right_odd_old
    new_odd = left_odd + right_even_old
    return left_even, left_odd, right_even_old, right_odd_old, new_even, new_odd

nums = [2, 1, 6, 4]
print(" i | Le Lo | Re Ro | chẵn mới lẻ mới | fair | kiểm tra trực tiếp")
for i in range(len(nums)):
    le, lo, re_, ro, ne, no = split_view(nums, i)
    rest = nums[:i] + nums[i + 1:]
    print(f" {i} | {le:2} {lo:2} | {re_:2} {ro:2} | {ne:8} {no:6} | {ne == no!s:5} | {is_fair(rest)}")

 i | Le Lo | Re Ro | chẵn mới lẻ mới | fair | kiểm tra trực tiếp
 0 |  0  0 |  6  5 |        5      6 | False | False
 1 |  2  0 |  6  4 |        6      6 | True  | True
 2 |  2  1 |  0  4 |        6      1 | False | False
 3 |  8  1 |  0  0 |        8      1 | False | False


## 5. Cách 1 — Brute Force (baseline)

**Ý tưởng:** làm đúng như đề nói. Với mỗi chỉ số `i`, tạo mảng mới không có `nums[i]`, tính tổng chẵn và tổng lẻ, kiểm tra bằng nhau.

- Tạo mảng mới: `O(n)`; tính hai tổng: `O(n)`; lặp `n` lần ⇒ **`O(n²)`**.
- Với `n = 10⁵` thì `10¹⁰` phép tính — chắc chắn **TLE**. Nhưng brute force cực kỳ có giá trị để **kiểm chứng** các cách nhanh.

In [8]:
def fair_brute(nums: List[int]) -> int:
    count = 0
    for i in range(len(nums)):
        rest = nums[:i] + nums[i + 1:]            # mảng sau khi xóa i
        if sum(rest[0::2]) == sum(rest[1::2]):
            count += 1
    return count

for nums, expected in examples:
    got = fair_brute(nums)
    print(nums, "->", got, "| kỳ vọng", expected)
    assert got == expected

[2, 1, 6, 4] -> 1 | kỳ vọng 1
[1, 1, 1] -> 3 | kỳ vọng 3
[1, 2, 3] -> 0 | kỳ vọng 0


**Độ phức tạp Cách 1:** Time `O(n²)`, Space `O(n)` (mảng `rest`).

**Nút thắt:** mỗi lần xóa ta tính lại **toàn bộ** hai tổng từ đầu, trong khi giữa hai lần xóa liên tiếp (`i` và `i+1`) mảng còn lại gần như y hệt nhau. Đo thời gian để thấy rõ tốc độ tăng bậc hai:

In [9]:
rnd = random.Random(1)
for n in (500, 1000, 2000):
    a = [rnd.randint(1, 10**4) for _ in range(n)]
    t = time.perf_counter(); fair_brute(a); dt = time.perf_counter() - t
    print(f"n={n:5}: {dt * 1000:7.1f} ms")
print("n tăng gấp đôi -> thời gian tăng ~4 lần (bậc hai). Với n = 1e5 sẽ mất hàng chục phút.")

n=  500:     4.2 ms
n= 1000:    15.7 ms
n= 2000:    46.7 ms
n tăng gấp đôi -> thời gian tăng ~4 lần (bậc hai). Với n = 1e5 sẽ mất hàng chục phút.


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** như mục 4, khi xóa `i`:

```
new_even(i) = LeftEven(i) + RightOdd(i)
new_odd(i)  = LeftOdd(i)  + RightEven(i)
```

trong đó `Left*` tính trên `nums[0..i-1]`, `Right*` tính trên `nums[i+1..n-1]` **theo chỉ số cũ**.

Cả bốn đại lượng đều là **tổng một đoạn theo chẵn/lẻ** ⇒ dùng tổng tiền tố là có trong `O(1)`.

**Quy tắc "nếu… thì…":**
- Nếu phần tử nằm **trái** chỗ xóa **thì** giữ nguyên vai trò chẵn/lẻ.
- Nếu phần tử nằm **phải** chỗ xóa **thì** đổi vai trò chẵn ↔ lẻ.
- Nếu `LeftEven + RightOdd == LeftOdd + RightEven` **thì** đếm chỉ số `i`.

**Invariant cho cách quét một lượt (Cách 3):** ngay trước khi xét `i`, `L = [LeftEven, LeftOdd]` là tổng của `nums[0..i-1]`, `R = [RightEven, RightOdd]` là tổng của `nums[i..n-1]`. Bỏ `nums[i]` khỏi `R` trước khi so sánh, rồi thêm vào `L` sau khi so sánh.

**Rút gọn thêm (Cách 4):** điều kiện tương đương `LeftEven - LeftOdd == RightEven - RightOdd`, tức **tổng đan dấu bên trái == tổng đan dấu bên phải** (cùng dấu theo chỉ số cũ). Chỉ cần một biến!

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Hai mảng tiền tố chẵn/lẻ

**Từng đoạn:**
1. Dựng `pe`, `po` độ dài `n+1` như mục 3.4.
2. Với mỗi `i`: `LeftEven = pe[i]`, `LeftOdd = po[i]`.
3. `RightEven = pe[n] - pe[i+1]`, `RightOdd = po[n] - po[i+1]` (bỏ qua chính `nums[i]`).
4. So sánh `LeftEven + RightOdd` với `LeftOdd + RightEven`.

Code demo (in ra bốn đại lượng):

In [10]:
def demo_prefix(nums):
    n = len(nums)
    pe, po = parity_prefix(nums)
    print("pe =", pe, "\npo =", po)
    for i in range(n):
        le, lo = pe[i], po[i]
        re_, ro = pe[n] - pe[i + 1], po[n] - po[i + 1]
        print(f"i={i}: Le={le}, Lo={lo}, Re={re_}, Ro={ro} -> chẵn mới={le + ro}, lẻ mới={lo + re_}, fair={le + ro == lo + re_}")

demo_prefix([2, 1, 6, 4])

pe = [0, 2, 2, 8, 8] 
po = [0, 0, 1, 1, 5]
i=0: Le=0, Lo=0, Re=6, Ro=5 -> chẵn mới=5, lẻ mới=6, fair=False
i=1: Le=2, Lo=0, Re=6, Ro=4 -> chẵn mới=6, lẻ mới=6, fair=True
i=2: Le=2, Lo=1, Re=0, Ro=4 -> chẵn mới=6, lẻ mới=1, fair=False
i=3: Le=8, Lo=1, Re=0, Ro=0 -> chẵn mới=8, lẻ mới=1, fair=False


Code nộp LeetCode của Cách 2:

In [11]:
class SolutionPrefix:
    def waysToMakeFair(self, nums: List[int]) -> int:
        n = len(nums)
        pe = [0] * (n + 1)
        po = [0] * (n + 1)
        for j, x in enumerate(nums):
            pe[j + 1] = pe[j] + (x if j % 2 == 0 else 0)
            po[j + 1] = po[j] + (x if j % 2 == 1 else 0)
        ans = 0
        for i in range(n):
            new_even = pe[i] + (po[n] - po[i + 1])
            new_odd = po[i] + (pe[n] - pe[i + 1])
            if new_even == new_odd:
                ans += 1
        return ans

for nums, expected in examples:
    assert SolutionPrefix().waysToMakeFair(nums) == expected
print("Cách 2 qua 3 ví dụ của đề")

Cách 2 qua 3 ví dụ của đề


**Độ phức tạp Cách 2:** Time `O(n)` (hai vòng lặp tuyến tính), Space `O(n)` (hai mảng `pe`, `po`).

### 7.2 Cách 3 — Quét một lượt với hai cặp tổng chạy (`O(1)` bộ nhớ)

Không cần lưu cả mảng tiền tố: ta chỉ cần biết tại thời điểm `i` tổng **trái** và tổng **phải**.

**Từng đoạn:**
1. Khởi tạo `R = [tổng chẵn toàn mảng, tổng lẻ toàn mảng]`, `L = [0, 0]`.
2. Tại chỉ số `i` với giá trị `x`: `R[i % 2] -= x` → bây giờ `R` chỉ còn phần **sau** `i`.
3. Kiểm tra `L[0] + R[1] == L[1] + R[0]` → nếu đúng, `ans += 1`.
4. `L[i % 2] += x` → đưa `x` sang phần trái, chuẩn bị cho `i+1`.

Code demo (in `L`, `R` qua từng bước):

In [12]:
def demo_running(nums):
    R = [sum(nums[0::2]), sum(nums[1::2])]
    L = [0, 0]
    print("khởi tạo: L =", L, "R =", R)
    for i, x in enumerate(nums):
        R[i % 2] -= x
        ok = L[0] + R[1] == L[1] + R[0]
        print(f"i={i}, x={x}: sau khi bỏ x khỏi R: L={L}, R={R} -> {L[0] + R[1]} vs {L[1] + R[0]} fair={ok}")
        L[i % 2] += x

demo_running([2, 1, 6, 4])

khởi tạo: L = [0, 0] R = [8, 5]
i=0, x=2: sau khi bỏ x khỏi R: L=[0, 0], R=[6, 5] -> 5 vs 6 fair=False
i=1, x=1: sau khi bỏ x khỏi R: L=[2, 0], R=[6, 4] -> 6 vs 6 fair=True
i=2, x=6: sau khi bỏ x khỏi R: L=[2, 1], R=[0, 4] -> 6 vs 1 fair=False
i=3, x=4: sau khi bỏ x khỏi R: L=[8, 1], R=[0, 0] -> 8 vs 1 fair=False


Code nộp LeetCode của Cách 3:

In [13]:
class SolutionRunning:
    def waysToMakeFair(self, nums: List[int]) -> int:
        R = [sum(nums[0::2]), sum(nums[1::2])]
        L = [0, 0]
        ans = 0
        for i, x in enumerate(nums):
            R[i % 2] -= x                    # bỏ nums[i] khỏi phần phải
            if L[0] + R[1] == L[1] + R[0]:   # trái giữ, phải đảo chẵn/lẻ
                ans += 1
            L[i % 2] += x                    # đưa nums[i] sang phần trái
        return ans

for nums, expected in examples:
    assert SolutionRunning().waysToMakeFair(nums) == expected
print("Cách 3 qua 3 ví dụ của đề")

Cách 3 qua 3 ví dụ của đề


**Độ phức tạp Cách 3:** Time `O(n)`, Space `O(1)` (chỉ vài biến; lát cắt `nums[0::2]` tạm thời tốn `O(n)` — có thể thay bằng một vòng lặp nếu muốn `O(1)` tuyệt đối).

### 7.3 Cách 4 — Một biến tổng đan dấu

Biến đổi điều kiện fair:

```
LeftEven + RightOdd == LeftOdd + RightEven
⇔ LeftEven − LeftOdd == RightEven − RightOdd
⇔ A_left(i) == A_right(i)
```

với `A_left(i) = Σ_{j<i} s_j·nums[j]`, `A_right(i) = Σ_{j>i} s_j·nums[j]`, `s_j = +1` nếu `j` chẵn, `−1` nếu `j` lẻ (theo **chỉ số cũ**).

Gọi `T` là tổng đan dấu toàn mảng. Ta có `A_right(i) = T − A_left(i) − s_i·nums[i]`. Vậy chỉ cần **một biến** `left` chạy dọc mảng.

Code demo:

In [14]:
def demo_alt(nums):
    T = alt_sum(nums)
    left = 0
    print("T (tổng đan dấu toàn mảng) =", T)
    for i, x in enumerate(nums):
        sx = x if i % 2 == 0 else -x
        right = T - left - sx
        print(f"i={i}: A_left={left:3}, s_i*x={sx:3}, A_right={right:3} -> fair={left == right}")
        left += sx

demo_alt([2, 1, 6, 4])

T (tổng đan dấu toàn mảng) = 3
i=0: A_left=  0, s_i*x=  2, A_right=  1 -> fair=False
i=1: A_left=  2, s_i*x= -1, A_right=  2 -> fair=True
i=2: A_left=  1, s_i*x=  6, A_right= -4 -> fair=False
i=3: A_left=  7, s_i*x= -4, A_right=  0 -> fair=False


Code nộp LeetCode của Cách 4:

In [15]:
class SolutionAlt:
    def waysToMakeFair(self, nums: List[int]) -> int:
        total = 0
        for i, x in enumerate(nums):
            total += x if i % 2 == 0 else -x
        left = 0
        ans = 0
        for i, x in enumerate(nums):
            sx = x if i % 2 == 0 else -x
            if left == total - left - sx:    # A_left == A_right
                ans += 1
            left += sx
        return ans

for nums, expected in examples:
    assert SolutionAlt().waysToMakeFair(nums) == expected
print("Cách 4 qua 3 ví dụ của đề")

Cách 4 qua 3 ví dụ của đề


**Độ phức tạp Cách 4:** Time `O(n)`, Space `O(1)` thật sự (không tạo lát cắt). Điều kiện còn có thể viết `2·left + sx == total`.

## 8. TRACE từng bước

Trace Cách 3 trên `nums = [6, 1, 7, 4, 1]` (ví dụ minh họa trong đề, không có đáp án — đã tính bằng brute force bên dưới).

Bảng trạng thái (giá trị **sau** bước `R[i%2] -= x`, **trước** bước `L[i%2] += x`):

| i | x | L (chẵn, lẻ) | R (chẵn, lẻ) | chẵn mới = L0+R1 | lẻ mới = L1+R0 | fair? |
|---|---|---|---|---|---|---|
| 0 | 6 | (0, 0) | (8, 5) | 5 | 8 | không |
| 1 | 1 | (6, 0) | (8, 4) | 10 | 8 | không |
| 2 | 7 | (6, 1) | (1, 4) | 10 | 2 | không |
| 3 | 4 | (13, 1) | (1, 0) | 13 | 2 | không |
| 4 | 1 | (13, 5) | (0, 0) | 13 | 5 | không |

Hàm `trace_running` in đúng bảng này:

In [16]:
def trace_running(nums):
    R = [sum(nums[0::2]), sum(nums[1::2])]
    L = [0, 0]
    ans = 0
    print(f"khởi tạo L={L}, R={R}")
    print(" i |  x | L(chẵn,lẻ) | R(chẵn,lẻ) | chẵn mới | lẻ mới | fair | ans")
    for i, x in enumerate(nums):
        R[i % 2] -= x
        ne, no = L[0] + R[1], L[1] + R[0]
        if ne == no:
            ans += 1
        print(f" {i} | {x:2} | ({L[0]:3},{L[1]:3}) | ({R[0]:3},{R[1]:3}) | {ne:8} | {no:6} | {ne == no!s:5}| {ans}")
        L[i % 2] += x
    return ans

res = trace_running([6, 1, 7, 4, 1])
print("kết quả:", res, "| brute force:", fair_brute([6, 1, 7, 4, 1]))

khởi tạo L=[0, 0], R=[14, 5]
 i |  x | L(chẵn,lẻ) | R(chẵn,lẻ) | chẵn mới | lẻ mới | fair | ans
 0 |  6 | (  0,  0) | (  8,  5) |        5 |      8 | False| 0
 1 |  1 | (  6,  0) | (  8,  4) |       10 |      8 | False| 0
 2 |  7 | (  6,  1) | (  1,  4) |       10 |      2 | False| 0
 3 |  4 | ( 13,  1) | (  1,  0) |       13 |      2 | False| 0
 4 |  1 | ( 13,  5) | (  0,  0) |       13 |      5 | False| 0
kết quả: 0 | brute force: 0


Trace thêm ví dụ 1 của đề (`[2,1,6,4]`, đáp án 1) bằng Cách 4 để thấy hai cách cho cùng kết luận tại cùng chỉ số:

In [17]:
def trace_alt(nums):
    total = sum(x if i % 2 == 0 else -x for i, x in enumerate(nums))
    left = ans = 0
    print("total =", total)
    for i, x in enumerate(nums):
        sx = x if i % 2 == 0 else -x
        right = total - left - sx
        hit = left == right
        ans += hit
        print(f"i={i}: left={left:3}, sx={sx:3}, right={right:3}, fair={hit}, ans={ans}")
        left += sx
    return ans

assert trace_alt([2, 1, 6, 4]) == 1

total = 3
i=0: left=  0, sx=  2, right=  1, fair=False, ans=0
i=1: left=  2, sx= -1, right=  2, fair=True, ans=1
i=2: left=  1, sx=  6, right= -4, fair=False, ans=1
i=3: left=  7, sx= -4, right=  0, fair=False, ans=1


## 9. Code nộp LeetCode

Chọn Cách 3 (dễ giải thích, `O(n)` time, `O(1)` bộ nhớ phụ):

In [18]:
class Solution:
    def waysToMakeFair(self, nums: List[int]) -> int:
        right = [0, 0]
        for i, x in enumerate(nums):
            right[i % 2] += x
        left = [0, 0]
        ans = 0
        for i, x in enumerate(nums):
            right[i % 2] -= x
            if left[0] + right[1] == left[1] + right[0]:
                ans += 1
            left[i % 2] += x
        return ans

for nums, expected in examples:
    got = Solution().waysToMakeFair(nums)
    print(nums, "->", got)
    assert got == expected

[2, 1, 6, 4] -> 1
[1, 1, 1] -> 3
[1, 2, 3] -> 0


## 10. Vì sao thuật toán đúng

**Bổ đề (đổi chỉ số):** sau khi xóa `nums[i]`, phần tử có chỉ số cũ `j` có chỉ số mới `j` nếu `j < i`, và `j − 1` nếu `j > i`. Vì `j` và `j − 1` khác tính chẵn lẻ nên mọi phần tử bên phải **đổi** chẵn/lẻ; bên trái **giữ nguyên**.

Do đó tổng chẵn mới = `LeftEven + RightOdd`, tổng lẻ mới = `LeftOdd + RightEven` — đúng là đại lượng mà mọi cách 2, 3, 4 so sánh.

**Invariant của Cách 3:** trước khi xử lý `i`:
- `left = (Σ nums[j], j<i, j chẵn ; Σ nums[j], j<i, j lẻ)`,
- `right = (Σ nums[j], j≥i, j chẵn ; Σ nums[j], j≥i, j lẻ)`.

Ban đầu `i = 0`: `left = (0,0)`, `right` = tổng toàn mảng ⇒ đúng. Bước `right[i%2] -= x` làm `right` thành tổng trên `j > i`; so sánh đúng theo bổ đề; bước `left[i%2] += x` làm `left` thành tổng trên `j < i+1` ⇒ invariant đúng cho `i+1`. Mỗi chỉ số được xét đúng một lần ⇒ `ans` đếm đúng.

**Cách 4:** trừ hai vế của điều kiện cho nhau là biến đổi tương đương, và `A_right = T − A_left − s_i·nums[i]` vì `T = A_left + s_i·nums[i] + A_right`.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`.

| Cách | Time | Space phụ | Ghi chú |
|---|---|---|---|
| 1. Brute force | `O(n²)` | `O(n)` | tạo mảng mới mỗi lần xóa |
| 2. Mảng tiền tố chẵn/lẻ | `O(n)` | `O(n)` | hai mảng `n+1` phần tử |
| 3. Hai cặp tổng chạy | `O(n)` | `O(1)` | 2 lượt duyệt |
| 4. Tổng đan dấu | `O(n)` | `O(1)` | một biến `left` |

Không có khác biệt average/worst: mọi cách đều duyệt hết mảng bất kể dữ liệu. Giá trị lớn nhất của tổng là `10⁵ · 10⁴ = 10⁹` — Python không lo tràn số (ngôn ngữ khác vẫn vừa `int32`).

Đo thực tế với `n = 10⁵`:

In [19]:
rnd = random.Random(7)
big = [rnd.randint(1, 10**4) for _ in range(10**5)]
for name, sol in [("Cách 2", SolutionPrefix()), ("Cách 3", SolutionRunning()), ("Cách 4", SolutionAlt()), ("Nộp bài", Solution())]:
    t = time.perf_counter(); r = sol.waysToMakeFair(big); dt = time.perf_counter() - t
    print(f"{name}: kết quả={r}, {dt * 1000:.1f} ms")

Cách 2: kết quả=0, 21.8 ms


Cách 3: kết quả=0, 12.9 ms


Cách 4: kết quả=0, 11.5 ms


Nộp bài: kết quả=0, 14.0 ms


## 12. Bộ test

Gồm: ví dụ đề, edge case (1 phần tử, 2 phần tử, mọi phần tử bằng nhau), một số mảng tự dựng (đã kiểm bằng brute force — đối chiếu lại đề gốc nếu cần), và **so sánh ngẫu nhiên** mọi cách với brute force (seed cố định, `1 ≤ nums[i] ≤ 10⁴`).

In [20]:
ALL = [("brute", fair_brute),
       ("prefix", SolutionPrefix().waysToMakeFair),
       ("running", SolutionRunning().waysToMakeFair),
       ("alt", SolutionAlt().waysToMakeFair),
       ("submit", Solution().waysToMakeFair)]

def check(nums, expected):
    for name, f in ALL:
        got = f(list(nums))
        assert got == expected, (name, nums, got, expected)

# 1) ví dụ đề
for nums, expected in examples:
    check(nums, expected)

# 2) "bẫy" khi tự dựng test: đoán nhanh [3, 3] -> 2 (vì "hai phần tử bằng nhau")?
guess = 2
real = fair_brute([3, 3])
print("đoán:", guess, "| brute force:", real, "| mọi cách:", [f([3, 3]) for _, f in ALL])
print("Đoán sai! Xóa một phần tử còn [3]: tổng chẵn 3, tổng lẻ 0 -> không fair.")

đoán: 2 | brute force: 0 | mọi cách: [0, 0, 0, 0, 0]
Đoán sai! Xóa một phần tử còn [3]: tổng chẵn 3, tổng lẻ 0 -> không fair.


Bài học: **không đoán** đáp án kỳ vọng của test tự dựng — tính tay rồi đối chiếu với brute force trước khi đưa vào `assert`. Bộ edge case dưới đây đã làm đúng như vậy (là ví dụ tự dựng, đã kiểm bằng brute force — đối chiếu lại đề gốc nếu cần):

In [21]:
# edge case (đáp án đã tính tay và đối chiếu brute force)
edge = [
    ([5], 1),                 # mảng rỗng sau khi xóa: 0 == 0
    ([3, 3], 0),              # còn [3]: 3 != 0
    ([1, 2], 0),
    ([4, 4, 4, 4, 4], 5),     # còn 4 phần tử bằng nhau: 8 == 8
    ([4, 4, 4, 4], 0),        # còn 3 phần tử: 8 != 4
    ([10**4] * 7, 7),
]
for nums, expected in edge:
    assert fair_brute(nums) == expected
    check(nums, expected)
print("edge case OK")

edge case OK


In [22]:
# 3) so sánh ngẫu nhiên với brute force (seed cố định, đúng ràng buộc đề)
rnd = random.Random(2024)
cnt_nonzero = 0
for _ in range(3000):
    n = rnd.randint(1, 30)
    if rnd.random() < 0.5:
        nums = [rnd.randint(1, 3) for _ in range(n)]   # giá trị nhỏ -> nhiều trường hợp fair
    else:
        nums = [rnd.randint(1, 10**4) for _ in range(n)]
    expected = fair_brute(nums)
    cnt_nonzero += expected > 0
    check(nums, expected)
print("3000 test ngẫu nhiên OK; số test có đáp án > 0:", cnt_nonzero)

3000 test ngẫu nhiên OK; số test có đáp án > 0: 1161


In [23]:
# 4) test cấu trúc: mảng đối xứng kiểu [a, a, a, ...] có độ dài lẻ -> mọi chỉ số đều fair
for n in range(1, 40, 2):
    nums = [7] * n
    assert Solution().waysToMakeFair(nums) == n
# 5) test lớn: các cách nhanh phải trùng nhau
rnd = random.Random(99)
for _ in range(5):
    a = [rnd.randint(1, 2) for _ in range(10**5)]
    r = {f(a) for name, f in ALL if name != "brute"}
    assert len(r) == 1
print("test cấu trúc + test lớn OK")

test cấu trúc + test lớn OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên rằng bên phải **đổi** chẵn/lẻ

Tưởng rằng xóa phần tử thì chỉ cần trừ nó khỏi tổng chẵn hoặc tổng lẻ ban đầu.

In [24]:
def wrong_no_swap(nums):
    even, odd = sum(nums[0::2]), sum(nums[1::2])
    ans = 0
    for i, x in enumerate(nums):
        e, o = even, odd
        if i % 2 == 0:
            e -= x
        else:
            o -= x
        if e == o:                # SAI: phần tử bên phải đã đổi chẵn/lẻ
            ans += 1
    return ans

bad = [2, 1, 6, 4]
print("sai:", wrong_no_swap(bad), "| đúng:", Solution().waysToMakeFair(bad))

sai: 0 | đúng: 1


**Vì sao sai:** xóa chỉ số 1 của `[2,1,6,4]` thì `6` chuyển từ chẵn sang lẻ và `4` từ lẻ sang chẵn; công thức trên vẫn coi chúng như cũ nên bỏ sót. **Sửa:** tách trái/phải, `new_even = LeftEven + RightOdd`, `new_odd = LeftOdd + RightEven` (Cách 3).

### Lỗi 2 — Trừ `x` khỏi phần phải **sau** khi so sánh (lệch một phần tử)

In [25]:
def wrong_order(nums):
    R = [sum(nums[0::2]), sum(nums[1::2])]
    L = [0, 0]
    ans = 0
    for i, x in enumerate(nums):
        if L[0] + R[1] == L[1] + R[0]:    # SAI: R vẫn còn chứa nums[i]
            ans += 1
        R[i % 2] -= x
        L[i % 2] += x
    return ans

for bad in ([1, 1, 1], [2, 1, 6, 4]):
    print(bad, "sai:", wrong_order(bad), "| đúng:", Solution().waysToMakeFair(bad))

[1, 1, 1] sai: 0 | đúng: 3
[2, 1, 6, 4] sai: 0 | đúng: 1


**Vì sao sai:** lúc so sánh, phần tử bị xóa vẫn nằm trong `R` nên được tính vào tổng mới. **Sửa:** thứ tự đúng là *bỏ khỏi R → so sánh → thêm vào L*.

### Lỗi 3 — Đảo chẵn/lẻ cả phần **trái**

Có bạn nhớ "có đổi chẵn/lẻ" nhưng lại đổi nhầm phía trái (hoặc đổi cả hai phía):

In [26]:
def wrong_swap_left(nums):
    R = [sum(nums[0::2]), sum(nums[1::2])]
    L = [0, 0]
    ans = 0
    for i, x in enumerate(nums):
        R[i % 2] -= x
        if L[1] + R[1] == L[0] + R[0]:   # SAI: coi trái đổi, phải giữ (hoặc không đổi gì)
            ans += 1
        L[i % 2] += x
    return ans

for bad in ([2, 1, 6, 4], [1, 2, 3], [6, 1, 7, 4, 1]):
    print(bad, "sai:", wrong_swap_left(bad), "| đúng:", Solution().waysToMakeFair(bad))

[2, 1, 6, 4] sai: 0 | đúng: 1
[1, 2, 3] sai: 0 | đúng: 0
[6, 1, 7, 4, 1] sai: 0 | đúng: 0


**Vì sao sai:** điều kiện `L[1] + R[1] == L[0] + R[0]` tương đương "tổng lẻ cũ == tổng chẵn cũ sau khi bỏ x", tức là **không** đổi chẵn/lẻ phía nào — giống lỗi 1. Chỉ phía **phải** (chỉ số lớn hơn) mới lùi chỉ số. **Sửa:** `L[0] + R[1] == L[1] + R[0]`.

### Lỗi 4 — Dùng `sum(...)` trong vòng lặp (vô tình quay về `O(n²)`)

In [27]:
def slow_but_correct(nums):
    ans = 0
    for i in range(len(nums)):
        le = sum(nums[0:i:2]); lo = sum(nums[1:i:2])
        start_e = i + 1 if (i + 1) % 2 == 0 else i + 2
        start_o = i + 1 if (i + 1) % 2 == 1 else i + 2
        re_ = sum(nums[start_e::2]); ro = sum(nums[start_o::2])
        ans += le + ro == lo + re_
    return ans

rnd = random.Random(5)
a = [rnd.randint(1, 10**4) for _ in range(3000)]
t = time.perf_counter(); r1 = slow_but_correct(a); dt1 = time.perf_counter() - t
t = time.perf_counter(); r2 = Solution().waysToMakeFair(a); dt2 = time.perf_counter() - t
print(f"kết quả giống nhau: {r1 == r2}; chậm: {dt1 * 1000:.1f} ms, nhanh: {dt2 * 1000:.2f} ms (n=3000)")

kết quả giống nhau: True; chậm: 58.6 ms, nhanh: 0.52 ms (n=3000)


**Vì sao "sai":** kết quả đúng nhưng mỗi vòng gọi `sum` trên lát cắt ⇒ `O(n²)` ⇒ TLE với `n = 10⁵`. **Sửa:** duy trì tổng chạy (Cách 3) hoặc dùng mảng tiền tố (Cách 2).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề nói "**xóa đúng một phần tử** rồi kiểm tra một tính chất toàn cục" ⇒ nghĩ ngay **trái / phải + tiền tố / hậu tố**.
- Tính chất liên quan **chỉ số chẵn/lẻ** ⇒ xóa phần tử làm **đảo** chẵn/lẻ phía sau.
- Kỹ thuật "**R bớt dần, L thêm dần**" trong một vòng lặp: thay cho hai mảng prefix/suffix, tiết kiệm bộ nhớ.
- Điều kiện "hai tổng bằng nhau" thường gom được thành "**một hiệu bằng 0**" (tổng đan dấu) → ít biến hơn, ít lỗi hơn.
- Luôn có brute force `O(n²)` sẵn sàng để đối chiếu trên `n ≤ 30`.

## 15. Biến thể của bài

1. **LC 724 — Find Pivot Index / LC 1991 — Find the Middle Index in Array:** tổng trái == tổng phải (không đổi chẵn/lẻ).
2. **LC 238 — Product of Array Except Self:** tiền tố/hậu tố nhân, bỏ phần tử hiện tại.
3. **LC 2270 — Number of Ways to Split Array:** đếm vị trí cắt thỏa điều kiện tổng trái/phải.
4. **Đổi điều kiện:** đếm chỉ số sao cho sau khi xóa, `tổng chẵn − tổng lẻ == k` (bài tập 5).
5. **Trả về danh sách chỉ số** thay vì số lượng (bài tập 4).
6. **Xóa hai phần tử liền kề:** phần bên phải lùi 2 chỉ số ⇒ **không** đổi chẵn/lẻ — tự thử tính!

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | xóa thật, tính lại hai tổng | `O(n²)` | `O(n)` | kiểm chứng, `n` nhỏ |
| 2. Prefix chẵn/lẻ | `pe`, `po` + tổng đoạn `O(1)` | `O(n)` | `O(n)` | dễ hiểu, cần truy vấn nhiều lần |
| 3. L/R chạy | bỏ khỏi R → so sánh → thêm vào L | `O(n)` | `O(1)` | **nộp bài** |
| 4. Tổng đan dấu | `A_left == T − A_left − s_i·x` | `O(n)` | `O(1)` | gọn nhất, phỏng vấn |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `waysToMakeFair` của (a) `[1,1,1,1]`; (b) `[1,1,1,1,1]`; (c) `[1,2,1]`.

**Bài 2 (tính tay):** Với `nums = [6,1,7,4,1]`, tính tổng đan dấu `T` và giá trị `A_left` tại mỗi chỉ số `i = 0..4` (trước khi cộng `s_i·nums[i]`).

**Bài 3 (điền TODO):** Hoàn thiện `fair_count_prefix(nums)` dùng hai mảng `pe`, `po`.

**Bài 4 (tự viết lại):** Viết `fair_indices(nums)` trả về **danh sách** các chỉ số mà xóa đi thì mảng fair (theo thứ tự tăng dần).

**Bài 5 (mini challenge):** Viết `count_diff_k(nums, k)` đếm số chỉ số `i` sao cho sau khi xóa `nums[i]`, `tổng chẵn − tổng lẻ == k`. (Với `k = 0` chính là bài gốc.)

In [28]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def fair_count_prefix(nums):
    n = len(nums)
    pe = [0] * (n + 1)
    po = [0] * (n + 1)
    # TODO: điền pe, po
    ans = 0
    for i in range(n):
        # TODO: tính new_even, new_odd và cập nhật ans
        pass
    return ans

In [29]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def fair_indices(nums):
    # TODO
    pass

def count_diff_k(nums, k):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: xóa một phần tử khỏi mảng toàn số 1 thì còn mảng toàn số 1 độ dài `n−1`. Mức 2: mảng toàn số 1 fair ⇔ độ dài chẵn. Mức 3: (a) còn 3 phần tử ⇒ `0`; (b) còn 4 ⇒ `5`; (c) xóa giữa còn `[1,1]` fair, xóa đầu/cuối còn `[2,1]`/`[1,2]` ⇒ `1`.

**Bài 2.** Mức 1: `s = [+,−,+,−,+]`. Mức 2: `T = 6 − 1 + 7 − 4 + 1`. Mức 3: `T = 9`, `A_left` lần lượt `0, 6, 5, 12, 8`.

**Bài 3.** Mức 1: `pe[j+1] = pe[j] + (x nếu j chẵn)`. Mức 2: `new_even = pe[i] + po[n] − po[i+1]`. Mức 3: `new_odd = po[i] + pe[n] − pe[i+1]`.

**Bài 4.** Mức 1: giống Cách 3 nhưng `append(i)` thay vì `ans += 1`. Mức 2: kiểm tra bằng brute force. Mức 3: `len(fair_indices(a)) == waysToMakeFair(a)`.

**Bài 5.** Mức 1: tổng chẵn mới − tổng lẻ mới = `A_left − A_right`. Mức 2: `A_right = T − A_left − s_i·x`. Mức 3: điều kiện `2·A_left + s_i·x − T == k`.

## 19. Đáp án tham khảo cho bài tập

In [30]:
# Bài 1
b1 = [Solution().waysToMakeFair(a) for a in ([1, 1, 1, 1], [1, 1, 1, 1, 1], [1, 2, 1])]
print("Bài 1:", b1)
assert b1 == [0, 5, 1]
assert b1 == [fair_brute(a) for a in ([1, 1, 1, 1], [1, 1, 1, 1, 1], [1, 2, 1])]

# Bài 2
nums = [6, 1, 7, 4, 1]
T = alt_sum(nums)
A = []
left = 0
for i, x in enumerate(nums):
    A.append(left)
    left += x if i % 2 == 0 else -x
print("Bài 2: T =", T, "A_left =", A)
assert T == 9 and A == [0, 6, 5, 12, 8]

Bài 1: [0, 5, 1]
Bài 2: T = 9 A_left = [0, 6, 5, 12, 8]


In [31]:
# Bài 3
def fair_count_prefix_sol(nums):
    n = len(nums)
    pe = [0] * (n + 1)
    po = [0] * (n + 1)
    for j, x in enumerate(nums):
        pe[j + 1] = pe[j] + (x if j % 2 == 0 else 0)
        po[j + 1] = po[j] + (x if j % 2 == 1 else 0)
    ans = 0
    for i in range(n):
        new_even = pe[i] + po[n] - po[i + 1]
        new_odd = po[i] + pe[n] - pe[i + 1]
        ans += new_even == new_odd
    return ans

rnd = random.Random(31)
for _ in range(1000):
    a = [rnd.randint(1, 4) for _ in range(rnd.randint(1, 25))]
    assert fair_count_prefix_sol(a) == fair_brute(a)
print("Bài 3 OK")

Bài 3 OK


In [32]:
# Bài 4
def fair_indices_sol(nums):
    R = [sum(nums[0::2]), sum(nums[1::2])]
    L = [0, 0]
    res = []
    for i, x in enumerate(nums):
        R[i % 2] -= x
        if L[0] + R[1] == L[1] + R[0]:
            res.append(i)
        L[i % 2] += x
    return res

def fair_indices_brute(nums):
    return [i for i in range(len(nums)) if is_fair(nums[:i] + nums[i + 1:])]

assert fair_indices_sol([2, 1, 6, 4]) == [1]
assert fair_indices_sol([1, 1, 1]) == [0, 1, 2]
rnd = random.Random(32)
for _ in range(1000):
    a = [rnd.randint(1, 3) for _ in range(rnd.randint(1, 25))]
    assert fair_indices_sol(a) == fair_indices_brute(a)
print("Bài 4 OK")

Bài 4 OK


In [33]:
# Bài 5
def count_diff_k_sol(nums, k):
    T = sum(x if i % 2 == 0 else -x for i, x in enumerate(nums))
    left = ans = 0
    for i, x in enumerate(nums):
        sx = x if i % 2 == 0 else -x
        right = T - left - sx
        if left - right == k:          # chẵn mới − lẻ mới = A_left − A_right
            ans += 1
        left += sx
    return ans

def count_diff_k_brute(nums, k):
    c = 0
    for i in range(len(nums)):
        rest = nums[:i] + nums[i + 1:]
        c += sum(rest[0::2]) - sum(rest[1::2]) == k
    return c

rnd = random.Random(33)
for _ in range(1500):
    a = [rnd.randint(1, 4) for _ in range(rnd.randint(1, 20))]
    k = rnd.randint(-6, 6)
    assert count_diff_k_sol(a, k) == count_diff_k_brute(a, k)
    assert count_diff_k_sol(a, 0) == Solution().waysToMakeFair(a)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "xóa một phần tử, phía sau đổi chẵn/lẻ":**

```python
right = [0, 0]
for i, x in enumerate(nums):
    right[i % 2] += x              # ban đầu: mọi thứ ở bên phải
left = [0, 0]
ans = 0
for i, x in enumerate(nums):
    right[i % 2] -= x              # 1) bỏ phần tử bị xóa khỏi bên phải
    if left[0] + right[1] == left[1] + right[0]:   # 2) trái giữ, phải đảo
        ans += 1
    left[i % 2] += x               # 3) chuyển sang bên trái
return ans
```

**Template tổng đan dấu:** `if 2 * left + sx == total: ans += 1; left += sx` với `sx = ±x`.

**Checklist:**
- [ ] Phía **phải** chỗ xóa đổi chẵn/lẻ, phía **trái** giữ nguyên?
- [ ] Thứ tự: bỏ khỏi `right` → so sánh → thêm vào `left`?
- [ ] Mảng 1 phần tử trả `1` (mảng rỗng là fair)?
- [ ] Không gọi `sum(...)` trên lát cắt bên trong vòng lặp (tránh `O(n²)`)?
- [ ] Đã đối chiếu với brute force trên vài trăm mảng ngẫu nhiên nhỏ?
- [ ] Chữ ký đúng: `def waysToMakeFair(self, nums: List[int]) -> int` và có `from typing import List` khi chạy ngoài LeetCode?